In [4]:
import pandas as pd

In [20]:
import pandas as pd

df = pd.read_csv("/content/superstore_dataset2011-2015.csv", encoding="cp1252")

In [6]:
df = pd.read_csv("superstore_dataset2011-2015.csv", encoding="cp1252")

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Rows: 51290
Columns: 24


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,City,State,...,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Shipping Cost,Order Priority
0,42433,AG-2011-2040,1/1/2011,6/1/2011,Standard Class,TB-11280,Toby Braunhardt,Consumer,Constantine,Constantine,...,OFF-TEN-10000025,Office Supplies,Storage,"Tenex Lockers, Blue",408.300,2,0.0,106.140,35.46,Medium
1,22253,IN-2011-47883,1/1/2011,8/1/2011,Standard Class,JH-15985,Joseph Holt,Consumer,Wagga Wagga,New South Wales,...,OFF-SU-10000618,Office Supplies,Supplies,"Acme Trimmer, High Speed",120.366,3,0.1,36.036,9.72,Medium
2,48883,HU-2011-1220,1/1/2011,5/1/2011,Second Class,AT-735,Annie Thurman,Consumer,Budapest,Budapest,...,OFF-TEN-10001585,Office Supplies,Storage,"Tenex Box, Single Width",66.120,4,0.0,29.640,8.17,High
3,11731,IT-2011-3647632,1/1/2011,5/1/2011,Second Class,EM-14140,Eugene Moren,Home Office,Stockholm,Stockholm,...,OFF-PA-10001492,Office Supplies,Paper,"Enermax Note Cards, Premium",44.865,3,0.5,-26.055,4.82,High
4,22255,IN-2011-47883,1/1/2011,8/1/2011,Standard Class,JH-15985,Joseph Holt,Consumer,Wagga Wagga,New South Wales,...,FUR-FU-10003447,Furniture,Furnishings,"Eldon Light Bulb, Duo Pack",113.670,5,0.1,37.770,4.70,Medium


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Row ID          51290 non-null  int64  
 1   Order ID        51290 non-null  object 
 2   Order Date      51290 non-null  object 
 3   Ship Date       51290 non-null  object 
 4   Ship Mode       51290 non-null  object 
 5   Customer ID     51290 non-null  object 
 6   Customer Name   51290 non-null  object 
 7   Segment         51290 non-null  object 
 8   City            51290 non-null  object 
 9   State           51290 non-null  object 
 10  Country         51290 non-null  object 
 11  Postal Code     9994 non-null   float64
 12  Market          51290 non-null  object 
 13  Region          51290 non-null  object 
 14  Product ID      51290 non-null  object 
 15  Category        51290 non-null  object 
 16  Sub-Category    51290 non-null  object 
 17  Product Name    51290 non-null 

In [8]:
df.isnull().sum()

,0
Row ID,0
Order ID,0
Order Date,0
Ship Date,0
Ship Mode,0
Customer ID,0
Customer Name,0
Segment,0
City,0
State,0


In [9]:
df.duplicated().sum()

np.int64(0)

In [13]:

clean_df = df.copy()


text_columns = clean_df.select_dtypes(include=["object", "string"]).columns

for col in text_columns:
    clean_df[col] = clean_df[col].str.strip()

# Converted dates
clean_df["Order Date"] = pd.to_datetime(
    clean_df["Order Date"], errors="coerce"
)

clean_df["Ship Date"] = pd.to_datetime(
    clean_df["Ship Date"], errors="coerce"
)

# Converted numeric columns
numeric_columns = [
    "Sales",
    "Quantity",
    "Discount",
    "Profit",
    "Shipping Cost"
]

for col in numeric_columns:
    clean_df[col] = pd.to_numeric(
        clean_df[col], errors="coerce"
    )


clean_df = clean_df.drop_duplicates()


clean_df["Shipping Days"] = (
    clean_df["Ship Date"] - clean_df["Order Date"]
).dt.days

print("Cleaning completed.")
print("Rows:", len(clean_df))
print("Columns:", len(clean_df.columns))

Cleaning completed.
Rows: 51290
Columns: 25


In [14]:
print("Rows after cleaning:", len(clean_df))
print("Duplicate rows:", clean_df.duplicated().sum())
print("Missing values:")
print(clean_df.isnull().sum())

Rows after cleaning: 51290
Duplicate rows: 0
Missing values:
Row ID                0
Order ID              0
Order Date        31223
Ship Date         31456
Ship Mode             0
Customer ID           0
Customer Name         0
Segment               0
City                  0
State                 0
Country               0
Postal Code       41296
Market                0
Region                0
Product ID            0
Category              0
Sub-Category          0
Product Name          0
Sales                 0
Quantity              0
Discount              0
Profit                0
Shipping Cost         0
Order Priority        0
Shipping Days     37993
dtype: int64


In [15]:
print("Negative Sales:", (clean_df["Sales"] < 0).sum())
print("Invalid Quantity:", (clean_df["Quantity"] <= 0).sum())
print("Invalid Discount:", (
    (clean_df["Discount"] < 0) |
    (clean_df["Discount"] > 1)
).sum())

Negative Sales: 0
Invalid Quantity: 0
Invalid Discount: 0


In [16]:
print(
    "Invalid dates:",
    (
        clean_df["Order Date"].isna() |
        clean_df["Ship Date"].isna() |
        (clean_df["Ship Date"] < clean_df["Order Date"])
    ).sum()
)

Invalid dates: 37993


In [17]:
clean_df.shape

(51290, 25)

In [18]:
cleaned_file = "superstore_cleaned.csv"

clean_df.to_csv(cleaned_file, index=False)

print("Cleaned dataset exported successfully:", cleaned_file)

Cleaned dataset exported successfully: superstore_cleaned.csv
